# 04 — LLM-Pipeline 1: Strukturiertes JSON → natürlichsprachliche Erklärung

Das LLM erhält globale Feature-Importance und lokale SHAP-/EBM-Beiträge als **strukturiertes JSON**
und formuliert daraus eine deutsche Erklärung für nicht-technische Nutzer.

- **Vorteil:** Präzise, maschinenlesbare Eingabe.
- **Nachteil:** Kein visueller Kontext (keine Plots).

Pro Kombination (XGBoost/EBM × 5 Instanzen) wird eine Erklärung generiert
und in `results/pipeline04/` gespeichert.

In [1]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import INSTANCE_IDS, EXPLANATIONS_DIR, RESULTS_DIR
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY   = "poisson_log"
MODEL      = DEFAULT_MODEL
MAX_TOKENS = 600
OUT_DIR    = RESULTS_DIR / "pipeline04"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM-Modell:    {MODEL}")
print(f"Loss-Option:   {LOSS_KEY}")
print(f"Testinstanzen: {INSTANCE_IDS}")
print(f"Ausgabe:       {OUT_DIR}")

LLM-Modell:    claude-sonnet-4-6
Loss-Option:   poisson_log
Testinstanzen: [224, 580, 1041, 1481, 1677, 2058, 2510, 3543, 3847, 4454]
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/pipeline04


## 1 · System-Prompt

Der System-Prompt wird gecacht (Anthropic Prompt Caching). Da das Minimum
für einen Cache-Block 1 024 Tokens beträgt, enthält der System-Prompt neben
den Instruktionen auch das Feature-Schema mit vollständigen Beschreibungen —
so überschreitet er die Schwelle zuverlässig und wird bei allen Folge-Aufrufen
aus dem Cache gelesen.


In [2]:
SYSTEM_PROMPT = """Du bist ein Experte für erklärbare KI (XAI) und formulierst Vorhersageerklärungen
für Mitarbeitende eines Fahrradverleihs — ohne technischen Hintergrund.

=== DOMAIN-KONTEXT ===
Das Capital-Bikeshare-System in Washington D.C. verleiht Fahrräder stundenweise.
Zwei Modelle (XGBoost und EBM) sagen vorher, wie viele Fahrräder (cnt) in einer
bestimmten Stunde ausgeliehen werden. Beide Modelle wurden mit Poisson-Deviance-Loss
trainiert; die Beiträge liegen im Log-Raum vor — d.h. die Vorhersage ergibt sich
als exp(Basiswert + Summe aller Beiträge). Positive Beiträge erhöhen, negative
senken die Vorhersage multiplikativ.

=== FEATURE-SCHEMA ===
Folgende Eingabemerkmale werden verwendet:

  hr          – Stunde des Tages (0–23). Bestimmt Pendelverkehr vs. Freizeitnutzung.
                0–5: Nacht (kaum Betrieb), 7–9: Morgenspitze, 17–19: Abendspitze,
                10–16: gleichmäßige Auslastung tagsüber.

  temp        – Normalisierte Temperatur (Wert × 41 = °C). Starker positiver Einfluss;
                optimaler Bereich ca. 0.5–0.8 (20–33 °C). Bei Kälte (<0.2, <8 °C)
                und Hitze (>0.9, >37 °C) sinkt die Nachfrage.

  yr          – Jahr (0 = 2011, 1 = 2012). Repräsentiert das allgemeine Wachstum
                des Verleihs über die Zeit.

  weathersit  – Wetterlage (1 = klar/wenige Wolken, 2 = Nebel/bewölkt,
                3 = leichter Regen/Schnee, 4 = Starkregen/Gewitter).
                Klares Wetter erhöht, schlechtes Wetter senkt die Nachfrage stark.

  mnth        – Monat (1 = Januar, 12 = Dezember). Saisoneffekte: Frühling/Sommer
                (April–September) = hohe Nachfrage, Winter = niedrig.

  weekday     – Wochentag (0 = Sonntag, 6 = Samstag). Werktage (1–5) zeigen
                deutliche Pendlerspitzen, Wochenende (0, 6) eher gleichmäßige
                Freizeitnutzung über den Mittag.

  hum         – Normalisierte Luftfeuchtigkeit (Wert × 100 = %). Hohe Feuchtigkeit
                (>0.8, >80 %) reduziert die Nachfrage leicht.

  windspeed   – Normalisierte Windgeschwindigkeit (Wert × 67 = km/h). Starker Wind
                (>0.4, >27 km/h) schreckt Nutzer ab.

  holiday     – Feiertag (0 = nein, 1 = ja). An Feiertagen fehlen Pendler;
                die Gesamtnachfrage sinkt typischerweise, Freizeitnutzung steigt.

=== AUSGABEFORMAT ===
Strukturiere deine Antwort in genau drei Abschnitte — ohne Zwischenüberschriften,
fließend lesbar, ca. 150–250 Wörter insgesamt:

  [VORHERSAGE] Nenne die vorhergesagte Anzahl, vergleiche mit dem tatsächlichen
  Wert und bewerte die Güte kurz (gut/mäßig/schlecht getroffen).

  [TREIBER] Erkläre die zwei oder drei wichtigsten Einflussfaktoren in dieser
  Stunde — mit konkreten Werten und ihrer Wirkungsrichtung. Nutze Alltagssprache
  statt Fachbegriffe (keine SHAP-Werte, kein "Log-Raum").

  [EMPFEHLUNG] Leite eine oder zwei praktische Schlussfolgerungen für den Betrieb
  ab (z.B. Fahrradverfügbarkeit, Wartungsfenster, Preisgestaltung).

Schreibe ausschließlich auf Deutsch. Keine Aufzählungszeichen am Absatzanfang."""

print(f"System-Prompt: {len(SYSTEM_PROMPT)} Zeichen, ~{len(SYSTEM_PROMPT)//4} Tokens (geschätzt)")


System-Prompt: 3039 Zeichen, ~759 Tokens (geschätzt)


## 2 · Hilfsfunktionen

`build_context_string` übersetzt normalisierte Rohwerte in lesbare Angaben
(z.B. `temp=0.68` → `~27.9 °C`). Diese Kontextzeile wird dem JSON-Payload
als `human_readable_context` beigefügt, damit das Modell nicht selbst
denormalisieren muss.


In [3]:
WEEKDAYS  = ["Sonntag", "Montag", "Dienstag", "Mittwoch",
             "Donnerstag", "Freitag", "Samstag"]
MONTHS    = ["", "Januar", "Februar", "März", "April", "Mai", "Juni",
             "Juli", "August", "September", "Oktober", "November", "Dezember"]
WEATHER   = {1: "klar/wenige Wolken", 2: "Nebel/bewölkt",
             3: "leichter Regen/Schnee", 4: "Starkregen/Gewitter"}


def build_context_string(fv: dict) -> str:
    """Gibt eine menschenlesbare Zusammenfassung der Feature-Werte zurück."""
    parts = []
    if "hr" in fv:
        parts.append(f"{int(fv['hr']):02d}:00 Uhr")
    if "weekday" in fv:
        parts.append(WEEKDAYS[int(fv["weekday"])])
    if "mnth" in fv:
        parts.append(MONTHS[int(fv["mnth"])])
    if "yr" in fv:
        parts.append("2011" if int(fv["yr"]) == 0 else "2012")
    if "weathersit" in fv:
        parts.append(WEATHER.get(int(fv["weathersit"]), "unbekannt"))
    if "temp" in fv:
        parts.append(f"~{float(fv['temp']) * 41:.1f} °C")
    if "hum" in fv:
        parts.append(f"{float(fv['hum']) * 100:.0f} % Luftfeuchtigkeit")
    if "windspeed" in fv:
        parts.append(f"Wind {float(fv['windspeed']) * 67:.1f} km/h")
    if "holiday" in fv and int(fv["holiday"]) == 1:
        parts.append("Feiertag")
    return ", ".join(parts)


def load_explanations(model_name: str, instance_id: int) -> tuple[dict, dict]:
    g = json.loads(
        (EXPLANATIONS_DIR / f"global_{model_name}_{LOSS_KEY}.json").read_text()
    )
    l = json.loads(
        (EXPLANATIONS_DIR / f"local_{model_name}_{LOSS_KEY}_inst{instance_id}.json").read_text()
    )
    return g, l


def build_user_prompt(global_exp: dict, local_exp: dict, top_k: int = 5) -> str:
    fv = local_exp["feature_values"]
    payload = {
        "model": local_exp["model"],
        "metrics": {
            "rmse":             global_exp["metrics"]["rmse"],
            "r2":               global_exp["metrics"]["r2"],
            "poisson_deviance": global_exp["metrics"]["poisson_deviance"],
        },
        "global_top_features": [
            {"rank": e["rank"], "feature": e["feature"]}
            for e in global_exp["global_importance"][:top_k]
        ],
        "instance_id":           local_exp["instance_id"],
        "feature_values":        fv,
        "human_readable_context": build_context_string(fv),
        "y_true":                local_exp["y_true"],
        "prediction":            local_exp["prediction"],
        "top_contributions":     local_exp["contributions"][:6],
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# Beispiel-Prompt anzeigen
g, l = load_explanations("xgb", INSTANCE_IDS[0])
sample_prompt = build_user_prompt(g, l)
print(f"Beispiel User-Prompt ({len(sample_prompt)} Zeichen):")
print(sample_prompt)


Beispiel User-Prompt (1382 Zeichen):
{
  "model": "xgb",
  "metrics": {
    "rmse": 39.006035,
    "r2": 0.951838,
    "poisson_deviance": 7.061543
  },
  "global_top_features": [
    {
      "rank": 1,
      "feature": "hr"
    },
    {
      "rank": 2,
      "feature": "yr"
    },
    {
      "rank": 3,
      "feature": "temp"
    },
    {
      "rank": 4,
      "feature": "weekday"
    },
    {
      "rank": 5,
      "feature": "mnth"
    }
  ],
  "instance_id": 224,
  "feature_values": {
    "weathersit": 1.0,
    "mnth": 7.0,
    "hr": 20.0,
    "weekday": 4.0,
    "yr": 0.0,
    "holiday": 0.0,
    "temp": 0.8,
    "hum": 0.63,
    "windspeed": 0.194
  },
  "human_readable_context": "20:00 Uhr, Donnerstag, Juli, 2011, klar/wenige Wolken, ~32.8 °C, 63 % Luftfeuchtigkeit, Wind 13.0 km/h",
  "y_true": 270.0,
  "prediction": 286.8653,
  "top_contributions": [
    {
      "feature": "hr",
      "value": 20.0,
      "contribution": 0.287352
    },
    {
      "feature": "yr",
      "va

## 3 · LLM-Aufrufe

> **Voraussetzung:** `ANTHROPIC_API_KEY` muss gesetzt sein:
> ```bash
> export ANTHROPIC_API_KEY=sk-ant-...
> ```

In [4]:
results = []
total_in, total_out, total_cache = 0, 0, 0

for model_name in ["xgb", "ebm"]:
    for iid in INSTANCE_IDS:
        g_exp, l_exp = load_explanations(model_name, iid)
        user_prompt = build_user_prompt(g_exp, l_exp)

        t0 = time.time()
        response = ask_text(
            user_prompt,
            system=SYSTEM_PROMPT,
            model=MODEL,
            max_tokens=MAX_TOKENS,
            cache_system=True,
        )
        elapsed = time.time() - t0

        text  = response["content"][0]["text"]
        usage = response.get("usage", {})
        in_tok  = usage.get("input_tokens", 0)
        out_tok = usage.get("output_tokens", 0)
        cache_r = usage.get("cache_read_input_tokens", 0)
        total_in += in_tok; total_out += out_tok; total_cache += cache_r

        record = {
            "pipeline":    "04_json",
            "llm_model":   MODEL,
            "loss_key":    LOSS_KEY,
            "xai_model":   model_name,
            "instance_id": iid,
            "explanation": text,
            "elapsed_s":   round(elapsed, 2),
            "usage":       {"input_tokens": in_tok, "output_tokens": out_tok,
                            "cache_read_input_tokens": cache_r},
            "prediction":  l_exp["prediction"],
            "y_true":      l_exp["y_true"],
        }
        results.append(record)
        (OUT_DIR / f"{model_name}_inst{iid}.json").write_text(
            json.dumps(record, indent=2, ensure_ascii=False)
        )
        print(f"  {model_name.upper()} inst={iid:4d}  "
              f"pred={l_exp['prediction']:6.1f}  y={l_exp['y_true']:5.0f}  "
              f"in={in_tok}  out={out_tok}  cache={cache_r}  t={elapsed:.1f}s")

print(f"\nGesamt:  input={total_in}  output={total_out}  cache_gelesen={total_cache}")

  XGB inst= 224  pred= 286.9  y=  270  in=1837  out=503  cache=0  t=10.7s
  XGB inst= 580  pred=  12.1  y=    5  in=1836  out=487  cache=0  t=10.9s
  XGB inst=1041  pred= 208.2  y=  229  in=1836  out=524  cache=0  t=12.8s
  XGB inst=1481  pred= 108.3  y=  113  in=1839  out=529  cache=0  t=11.6s
  XGB inst=1677  pred= 180.3  y=  145  in=1839  out=523  cache=0  t=11.8s
  XGB inst=2058  pred= 179.5  y=  238  in=1839  out=505  cache=0  t=13.1s
  XGB inst=2510  pred= 368.9  y=  337  in=1838  out=507  cache=0  t=12.6s
  XGB inst=3543  pred= 660.0  y=  691  in=1838  out=536  cache=0  t=12.5s
  XGB inst=3847  pred= 141.4  y=  122  in=1836  out=529  cache=0  t=16.9s
  XGB inst=4454  pred= 382.3  y=  311  in=1838  out=561  cache=0  t=12.5s
  EBM inst= 224  pred= 290.1  y=  270  in=1835  out=446  cache=0  t=13.4s
  EBM inst= 580  pred=   9.7  y=    5  in=1836  out=480  cache=0  t=11.7s
  EBM inst=1041  pred= 221.3  y=  229  in=1835  out=517  cache=0  t=10.8s
  EBM inst=1481  pred= 124.9  y=  113 

## 4 · Beispiel-Erklärungen

In [5]:
for rec in results[:2]:
    sep = '=' * 70
    print(sep)
    print(f"Modell: {rec['xai_model'].upper()}  |  Instanz: {rec['instance_id']}  "
          f"|  Vorhersage: {rec['prediction']:.1f}  |  Tatsächlich: {rec['y_true']:.0f}")
    print(sep)
    print(rec["explanation"])
    print()

Modell: XGB  |  Instanz: 224  |  Vorhersage: 286.9  |  Tatsächlich: 270
[VORHERSAGE] Das Modell sagte für diese Stunde 287 Ausleihen vorher; tatsächlich wurden 270 Fahrräder ausgeliehen. Die Abweichung beträgt rund 17 Räder, was einem Fehler von etwa 6 Prozent entspricht. Das ist ein gutes Ergebnis — das Modell trifft die reale Nachfrage hier sehr zuverlässig.

[TREIBER] Den stärksten Einfluss hat die Uhrzeit: 20 Uhr ist der Abend nach der Hauptverkehrszeit, in der viele Pendler noch heimfahren und Freizeitnutzer unterwegs sind — das treibt die Nachfrage spürbar nach oben. Die warme Temperatur von knapp 33 °C verstärkt diesen Effekt zusätzlich, da angenehme Sommerabende erfahrungsgemäß viele Menschen zum Radfahren einladen. Auch der Monat Juli trägt positiv bei: Der Hochsommer gehört zur nachfragestärksten Jahreszeit. Leicht gebremst wird die Vorhersage allerdings dadurch, dass die Daten aus dem Jahr 2011 stammen — das System war damals noch nicht so bekannt und genutzt wie im Folgejah

## 5 · Zusammenfassung

In [6]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Modell":     r["xai_model"].upper(),
        "Instanz":    r["instance_id"],
        "y_true":     r["y_true"],
        "Vorhersage": r["prediction"],
        "Wörter":     len(r["explanation"].split()),
        "tok_input":  r["usage"]["input_tokens"],
        "tok_output": r["usage"]["output_tokens"],
        "Zeit (s)":   r["elapsed_s"],
    }
    for r in results
])
display(summary)

,Modell,Instanz,y_true,Vorhersage,Wörter,tok_input,tok_output,Zeit (s)
0,XGB,224,270.0,286.8653,202,1837,503,10.73
1,XGB,580,5.0,12.1144,209,1836,487,10.91
2,XGB,1041,229.0,208.2260,215,1836,524,12.80
3,XGB,1481,113.0,108.3289,219,1839,529,11.55
4,XGB,1677,145.0,180.2743,208,1839,523,11.77
5,XGB,2058,238.0,179.4730,202,1839,505,13.13
6,XGB,2510,337.0,368.9001,204,1838,507,12.56
7,XGB,3543,691.0,660.0004,222,1838,536,12.52
8,XGB,3847,122.0,141.3505,211,1836,529,16.91
9,XGB,4454,311.0,382.2661,231,1838,561,12.50
